# Chapter 5: Formatting Output & Structured Responses

**Technique:** Output formatting and structured responses

**Contents**
* [Lesson: Controlling Output Shape](#lesson)
* [Exercises](#exercises)
* [Example Playground](#example-playground)

In [1]:
# Make the project root importable from any working directory, then load the helpers.
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "lib" / "helpers.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lib.helpers import get_completion, response_text, client, MODEL, MAX_TOKENS, SAMPLING_PARAMS

<a id="lesson"></a>
## Lesson: Controlling Output Shape

When you consume Claude's output in a program, not just display it to a human, you need the response to arrive in a predictable shape. Four techniques give you that control:

### 1. Explicit "no preamble" instructions

By default Claude may open with a sentence like "Sure, here's my review…". Tell it not to:

```python
PROMPT = "Review this code. Skip any preamble, output only the review.\nlast = items[len(items)]"
```

This works for any format: JSON, markdown tables, bullet lists. The instruction travels inside the user message, so no extra API parameters are needed.

### 2. XML tags for structured prose

Ask Claude to wrap sections in XML tags and parse (or highlight) them downstream:

```python
PROMPT = "Review this line and wrap your answer in <review></review> tags.\nlast = items[len(items)]"
```

Tags give you a reliable string boundary to extract with a regex (`re.search(r"<review>(.*?)</review>", text, re.S)`) or a simple `split`. They also let you request *multiple named sections* in one call, for example `<summary>` and `<risks>`.

### 3. Structured outputs via `output_config`

For machine readable payloads, use the `output_config` parameter with a JSON Schema. The API guarantees the response text is valid JSON that conforms to your schema:

```python
import json

msg = client.messages.create(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    messages=[{"role": "user", "content": "Extract issues from this review note: 'off-by-one in last_item; also missing None check'."}],
    output_config={
        "format": {
            "type": "json_schema",
            "schema": {
                "type": "object",
                "properties": {
                    "summary": {"type": "string"},
                    "issues": {"type": "array", "items": {"type": "string"}},
                },
                "required": ["summary", "issues"],
                "additionalProperties": False,
            },
        },
    },
)
data = json.loads(response_text(msg))
```

Important details:
* Call `client.messages.create` directly (not the `get_completion` helper) so you can pass `output_config`.
* Always include `"additionalProperties": False`; without it the schema is underspecified and the model may add unexpected fields.
* The response is still a string; call `json.loads` to get a `dict`. Read it with `response_text(msg)` rather than `msg.content[0].text`: on current models the first content block can be a `thinking` block.

**Pydantic models.** The Python SDK can derive the schema from a Pydantic model and validate the result for you with `client.messages.parse(..., output_format=MyModel)`; the validated object is on `response.parsed_output`. There is an example in the cells below.

### 4. Stop sequences

Pass a `stop_sequences` list to truncate the response at a sentinel string. Useful when you want only the first section of a longer potential output:

```python
msg = client.messages.create(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    messages=[{"role": "user", "content": "Review this line inside <review></review> tags, then suggest a fix: last = items[len(items)]"}],
    stop_sequences=["</review>"],
)
print(msg.stop_reason, msg.stop_sequence)  # "stop_sequence", "</review>"
```

### Historical note: assistant turn prefilling

Older Anthropic models accepted a partial `assistant` turn at the end of the messages list, letting you "speak for Claude" and force a specific opening. On current models (Sonnet 4.6 and later, including the course default `claude-opus-5-5`) that pattern returns a **400 Bad Request**, so the four techniques above replace it entirely.

In [2]:
print(get_completion("Review this line and wrap your answer in <review></review> tags: last = items[len(items)]"))

<review>
**Bug: off-by-one error that always raises `IndexError`.**

Python sequences are zero-indexed, so valid indices run from `0` to `len(items) - 1`. `items[len(items)]` is always one past the end and fails on every call, including on empty lists.

**Fix (idiomatic):**
```python
last = items[-1]
```
Negative indexing counts from the end, so `-1` is the last element. This is clearer than the explicit alternative, `items[len(items) - 1]`.

**Edge case: empty sequences.**
Both versions still raise `IndexError` if `items` is empty. If that can happen, handle it explicitly:
```python
last = items[-1] if items else None   # or a suitable default
```
Or, if an empty list is truly an error, let the exception propagate, or raise a more descriptive one:
```python
if not items:
    raise ValueError("items must not be empty")
last = items[-1]
```

**Minor notes:**
- If `items` is an iterator or generator rather than a list, neither `len()` nor indexing will work. Use something like `collectio

In [3]:
import json

msg = client.messages.create(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    messages=[{"role": "user", "content": "Extract issues from this review note: 'off-by-one in last_item; also missing None check'."}],
    output_config={
        "format": {
            "type": "json_schema",
            "schema": {
                "type": "object",
                "properties": {
                    "summary": {"type": "string"},
                    "issues": {"type": "array", "items": {"type": "string"}},
                },
                "required": ["summary", "issues"],
                "additionalProperties": False,
            },
        },
    },
    **SAMPLING_PARAMS,
)
data = json.loads(response_text(msg))
print(type(data))
print(json.dumps(data, indent=2))

<class 'dict'>
{
  "summary": "Review note identifies two bugs: an off-by-one error in last_item and a missing None check.",
  "issues": [
    "Off-by-one error in last_item",
    "Missing None check"
  ]
}


In [4]:
# Pydantic: let a model class define the schema and validate the response.
from pydantic import BaseModel


class ReviewNote(BaseModel):
    summary: str
    issues: list[str]


parsed = client.messages.parse(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    messages=[{"role": "user", "content": "Extract issues from this review note: 'off-by-one in last_item; also missing None check'."}],
    output_format=ReviewNote,
    **SAMPLING_PARAMS,
)
note = parsed.parsed_output  # a validated ReviewNote instance
print(note.summary)
for issue in note.issues:
    print("-", issue)

Review identified an off-by-one error in last_item and a missing None check.
- Off-by-one error in last_item
- Missing None check


In [5]:
# Stop sequences: generation halts as soon as Claude emits the sentinel string.
msg = client.messages.create(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    messages=[{"role": "user", "content": "Review this line. Put the review in <review></review> tags, then add a <fix></fix> section with corrected code: last = items[len(items)]"}],
    stop_sequences=["</review>"],
    **SAMPLING_PARAMS,
)
print(response_text(msg))
print("\nstop_reason:", msg.stop_reason, "| stop_sequence:", msg.stop_sequence)

<review>
**Bug: off-by-one error that always raises `IndexError`.**

Python lists are zero-indexed, so valid indices run from `0` to `len(items) - 1`. `len(items)` is always one past the last valid index, so `items[len(items)]` fails on every list, including empty ones.

**Secondary concern: empty sequences.**
Even after fixing the index, accessing the last element of an empty list raises `IndexError`. If `items` can be empty, handle that case explicitly.

**Style:**
Python supports negative indexing, so `items[-1]` is the idiomatic way to get the last element. It's clearer and avoids manual length arithmetic.


stop_reason: stop_sequence | stop_sequence: </review>


<a id="exercises"></a>
## Exercises

### Exercise 5.1: Wrap a code review in `<review>` tags

**Scenario**: your CI pipeline parses Claude's code review output by looking for a `<review>` XML tag. The grader simulates that parser.

**Task**: write a prompt that asks Claude to review this line, `last = items[len(items)]`, and wrap its answer in `<review></review>` tags.

**Success criteria**: Claude's response contains the string `<review>`.

In [6]:
from lib.grading import includes_any, grade

PROMPT = "Review this line and wrap your entire answer in <review></review> tags: last = items[len(items)]"

response = get_completion(PROMPT)

def grade_exercise(text):
    return includes_any(text, ["<review>"])

grade(response, grade_exercise(response))

<review>
**Bug: off-by-one error.** This line always raises `IndexError`.

Python lists are zero-indexed, so valid indices run from `0` to `len(items) - 1`. Index `len(items)` is one past the end. For example, with `items = [10, 20, 30]`, `len(items)` is `3`, but the last index is `2`.

**Fix (idiomatic):**
```python
last = items[-1]
```

Negative indexing counts from the end, so `-1` is the last element. Avoid the more verbose `items[len(items) - 1]`.

**Empty lists:** `items[-1]` still raises `IndexError` if `items` is empty. If that can happen, guard it:
```python
last = items[-1] if items else None
```

Or let the exception propagate if an empty list means a bug upstream. The right choice depends on how the caller should handle that case.

**Other types:** If `items` is a dict rather than a list, `items[len(items)]` would look up the key equal to the length. That is almost certainly not what you want, so confirm the type.
</review>

--------------------------- GRADING -------------

In [7]:
from hints import exercise_5_1_hint
print(exercise_5_1_hint)

The grader looks for the XML tag <review> in the output. Ask Claude to wrap its answer in that tag.


### Exercise 5.2: Two delimited sections: `<summary>` and `<risks>`

**Scenario**: a security review tool expects two separate XML sections (a brief summary and a risk list) so each can be routed to a different downstream handler.

**Task**: write a prompt that asks Claude to review the code snippet `requests.get(user_input)` and return **two sections**: one wrapped in `<summary></summary>` tags and one wrapped in `<risks></risks>` tags.

**Success criteria**: Claude's response contains both `<summary>` and `<risks>`.

In [8]:
from lib.grading import includes_all, grade

PROMPT = "Review the Python code snippet requests.get(user_input). Return exactly two sections: a brief overview wrapped in <summary></summary> tags, and a list of security and correctness risks wrapped in <risks></risks> tags."

response = get_completion(PROMPT)

def grade_exercise(text):
    return includes_all(text, ["<summary>", "<risks>"])

grade(response, grade_exercise(response))

<summary>
The snippet `requests.get(user_input)` performs an HTTP GET request to a URL supplied directly by the user. There is no validation, sanitization, timeout, error handling, or response checking. This pattern is common in URL fetchers, webhook handlers, and preview generators. Without safeguards it is a classic Server-Side Request Forgery (SSRF) vector and is fragile in production.
</summary>

<risks>
**Security risks**

1. **Server-Side Request Forgery (SSRF):** An attacker can make the server send requests to internal resources it can reach but the attacker cannot. Examples:
   - `http://localhost`, `http://127.0.0.1:6379`, `http://10.0.0.5/admin`
   - Cloud metadata endpoints such as `http://169.254.169.254/latest/meta-data/`, which can expose IAM credentials on AWS, GCP, or Azure.
   - Internal service discovery and port scanning, using response timing or error differences.

2. **Redirect-based bypass:** `requests` follows redirects by default (`allow_redirects=True`). Valid

In [9]:
from hints import exercise_5_2_hint
print(exercise_5_2_hint)

Ask for two clearly delimited sections. The grader checks both <summary> and <risks> tags appear.


### Exercise 5.3: Structured output with `output_config`

**Scenario**: a code analysis pipeline needs machine readable output it can pass directly to a database insert, with no postprocessing regex and no fragile text parsing.

**Task**: fill in the `schema` dict so Claude's response parses to a Python `dict` with:
* `summary`, a `string` field
* `issues`, an `array` of strings

The `USER` message and the API call are already wired up; you only need to replace the `schema` placeholder. (While `schema` is still empty the API rejects the request; the cell catches that error and grades it as not yet solved.)

**Success criteria**: `json.loads(response_text(msg))` produces a dict where `isinstance(obj["summary"], str)` and `isinstance(obj["issues"], list)` are both true.

In [10]:
import json

import anthropic
from lib.grading import grade

USER = "Summarize and list the problems in: 'no input validation; SQL string concatenation; no tests'."
schema = {
    "type": "object",
    "properties": {
        "summary": {"type": "string"},
        "issues": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["summary", "issues"],
    "additionalProperties": False,
}

text, parsed = "", None
try:
    msg = client.messages.create(
        model=MODEL, max_tokens=MAX_TOKENS,
        messages=[{"role": "user", "content": USER}],
        output_config={"format": {"type": "json_schema", "schema": schema}},
        **SAMPLING_PARAMS,
    )
    text = response_text(msg)
    parsed = json.loads(text)
except (anthropic.BadRequestError, json.JSONDecodeError) as err:
    text = text or f"Request failed: {err}"

def grade_exercise(obj):
    return isinstance(obj, dict) and isinstance(obj.get("summary"), str) and isinstance(obj.get("issues"), list)

grade(text, grade_exercise(parsed))

{"summary":"The code has three significant quality and security problems. It does not validate input, it builds SQL queries by concatenating strings, which exposes it to SQL injection, and it has no automated tests.","issues":["No input validation: user-supplied data is not checked for type, format, length, or range. This can lead to unexpected behavior, crashes, or security vulnerabilities.","SQL string concatenation: building queries by concatenating strings exposes the application to SQL injection attacks. Use parameterized queries or prepared statements instead.","No tests: the absence of automated tests makes it hard to verify correctness, catch regressions, and refactor safely."]}

--------------------------- GRADING ---------------------------
This exercise has been correctly solved: True


In [11]:
from hints import exercise_5_3_hint
print(exercise_5_3_hint)

Use output_config.format with a json_schema so the response is guaranteed-parseable JSON. The grader parses it and checks for a "summary" string and an "issues" list.


## Common mistakes, stretch challenge, and reflection

**Common mistakes**

* **Trying to prefill the assistant turn**: passing `{"role": "assistant", "content": "..."}` as the last message returns a **400 Bad Request** on current models. Use explicit instructions or `output_config` instead.
* **Forgetting `"additionalProperties": False`**: without it the JSON Schema is underspecified. Claude may add extra fields that break downstream code expecting exactly the fields you declared.
* **Not calling `json.loads`**: the response text is always a `str`. Even when structured output is enabled, you must parse it yourself before accessing keys (or use `client.messages.parse` with a Pydantic model).
* **Reading `msg.content[0].text`**: on current models `content[0]` may be a `thinking` block with no `.text`. Select the text blocks (`response_text(msg)`).

**Stretch challenge**

Extend the Exercise 5.3 schema to include a `severity` field with an `enum` constraint (`["low", "medium", "high"]`). Add `"severity"` to the `required` list, rerun, and observe how Claude populates it. Then do the same with a Pydantic model using `Literal["low", "medium", "high"]`.

**Reflect**: when is structured output (`output_config`) better than asking Claude to return JSON in free text and parsing it with a regex? Consider error handling, schema evolution, and the cost of a malformed response in a production pipeline.

<a id="example-playground"></a>
## Example Playground

Use the cells below to experiment freely. Try combining XML tags with `output_config`, add stop sequences, or request additional schema fields.

In [12]:
# Experiment: structured output with an extra "severity" field
import json

schema = {
    "type": "object",
    "properties": {
        "summary": {"type": "string"},
        "issues": {"type": "array", "items": {"type": "string"}},
        "severity": {"type": "string", "enum": ["low", "medium", "high"]},
    },
    "required": ["summary", "issues", "severity"],
    "additionalProperties": False,
}

msg = client.messages.create(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    messages=[{"role": "user", "content": 'Analyze this Flask code: return requests.get(request.args["url"]).text'}],
    output_config={"format": {"type": "json_schema", "schema": schema}},
    **SAMPLING_PARAMS,
)
print(json.dumps(json.loads(response_text(msg)), indent=2))

{
  "summary": "The code fetches whatever URL the user supplies in the 'url' query parameter and returns the response body. This is a classic Server-Side Request Forgery (SSRF) vulnerability: an attacker can make the server send requests to internal services, cloud metadata endpoints, or arbitrary external hosts, then read the responses.",
  "issues": [
    "Server-Side Request Forgery (SSRF): unvalidated user-controlled URL passed directly to requests.get, allowing access to internal network resources (e.g., http://localhost, 10.x.x.x, 192.168.x.x).",
    "Cloud metadata exposure: attackers can query endpoints like http://169.254.169.254/latest/meta-data/ to steal cloud credentials (AWS/GCP/Azure).",
    "Non-HTTP scheme abuse: depending on installed adapters, other schemes could be used; there is no restriction to http/https.",
    "No domain allowlist and no protection against DNS rebinding or redirect-based bypasses (requests follows redirects by default).",
    "Reflected content 